# Deep Convolutional Q-Learning for Pac-Man

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!pip install ale-py
!apt-get install -y swig
!pip install gymnasium[box2d]

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from torch.utils.data import DataLoader, TensorDataset

### Creating the architecture of the Neural Network

In [ ]:
class Network(nn.Module): #Inheriting the nn.Module class in out network class

  def __init__(self,action_size, seed = 42): #state_size is not required here as state will be images
    super(Network,self).__init__()
    self.seed = torch.manual_seed(seed)

    #Implementing Convolutional Layer

    self.conv1 = nn.Conv2d(3,32,8,stride=4) #input_channels(rgb image so 3),output_channels(default value),kernel_size(default)
    self.bn1 = nn.BatchNorm2d(num_features=32)  # batch normalization layer (32 because its the output of previous layer)

    self.conv2 = nn.Conv2d(32,64,4,stride=2)
    self.bn2 = nn.BatchNorm2d(num_features=64)

    self.conv3 = nn.Conv2d(64,64,3,stride=1)
    self.bn3 = nn.BatchNorm2d(num_features=64)

    self.conv4 = nn.Conv2d(64,128,3,stride=1)
    self.bn4 = nn.BatchNorm2d(num_features=128)

    #Full Connection layers below
    self.fc1 = nn.Linear(10*10*128,512) # 10*10*128 these values are calculated using a formula as output of convolutional layer is flattened
    self.fc2 = nn.Linear(512,256)
    self.fc3 = nn.Linear(256,action_size)

  def forward(self,state): #state is current state of environment at any time
    # Below 2 lines of code propogate the signal from input layer to first fully connected layer

    # below line of code takes game screen image passes it to 1st conolutional layer then to first batch normalization layer then to relu function
    x = F.relu(self.bn1(self.conv1(state))) # the state here is the image of the game screen
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))

    # Below lines will flatten the tensor/vector to fit it into the fully connected layed. This is a part of Convolution
    x = x.view(x.size(0),-1)

    # Below 2 lines of code propogate the signal from Convolution layers to fully connected layers
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x) # Removed redundant call and ReLU activation

    return x

### Setting up the environment

In [ ]:
import ale_py
import gymnasium as gym
env = gym.make('MsPacmanNoFrameskip-v0', full_action_space = False) # The MsPacman environment was renamed 'MsPacmanNoFrameskip-v0'
state_shape = env.observation_space.shape
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape: ', state_shape) # size of image (height,width,rgb)
print('State size: ', state_size)
print('Number of actions: ', number_actions)

### Initializing the hyperparameters

In [ ]:
learning_rate = 5e-4 # (e = 10^)
minibatch_size = 64
discount_factor = 0.99 # value close to 1 makes the model consider future rewards and improves performance but not in all cases so be carefull

### Preprocessing the frames

In [ ]:
from PIL import Image
from torchvision import transforms

def preprocess_frame(frame):
  frame = Image.fromarray(frame) # frame here is image in form of numpy array and this function makes it to a PIL image object
  preprocess = transforms.Compose([
      transforms.Resize((128,128)), # we do this because original image is of size (210,160) and if we use this it will take a long time to train model because of larger size
      transforms.ToTensor(), # Converts PIL Image object to a tensor so we feed it to the network and also normalizes the pixel to a range(0,1)
  ])

  return preprocess(frame).unsqueeze(0) # This is used to add extra dimension to vector which is just the batch number

### Implementing the DCQN class

In [ ]:
class Agent():
  def __init__(self, action_size):
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    self.action_size = action_size # this is the same variable as number_of_action
    self.local_qnetwork = Network(action_size).to(self.device) # this is a q network which selects the action of agent
    self.target_qnetwork = Network(action_size).to(self.device) # this q network calculates the target q values which will be used to train local q_network
    self.optimizer = optim.Adam(self.local_qnetwork.parameters(),lr = learning_rate)
    self.memory = deque(maxlen = 10000) # for storing learning experiences


 # Step function below stores the experiences and decides when to learn from them
  def step(self, state, action, reward, next_state, done) :
    # The inputs state etc are numpy arrays from the game(images are numpy arrays) so we first process them then feed them to neural network
    state = preprocess_frame(state)
    next_state = preprocess_frame(next_state)
    self.memory.append((state, action, reward, next_state, done))
    if len(self.memory) > minibatch_size:
      experiences = random.sample(self.memory,minibatch_size) # takes a random sample of memories from self.memory
      self.learn(experiences,discount_factor)

 # Below function selects the appropriate action to perform (epsilon greedy action selection process)
  def act(self,state,epsilon = 0.):
    state = preprocess_frame(state).to(self.device)
    self.local_qnetwork.eval()
    with torch.no_grad():
      action_values = self.local_qnetwork(state)
    self.local_qnetwork.train()
    if random.random() > epsilon:
      return np.argmax(action_values.cpu().data.numpy())
    else:
      return random.choice(np.arange(self.action_size))

  def learn(self, experiences, discount_factor):
    states, actions, rewards, next_states, dones = zip(*experiences)

    # Convert tuples of data to tensors
    states_batch = torch.cat(states).float().to(self.device)
    actions_batch = torch.from_numpy(np.array(actions)).long().unsqueeze(1).to(self.device)
    rewards_batch = torch.from_numpy(np.array(rewards)).float().unsqueeze(1).to(self.device)
    next_states_batch = torch.cat(next_states).float().to(self.device)
    dones_batch = torch.from_numpy(np.array(dones).astype(np.uint8)).float().unsqueeze(1).to(self.device)

    # Get max predicted Q values (for next states) from target model
    next_q_targets = self.target_qnetwork(next_states_batch).detach().max(1)[0].unsqueeze(1)
    # Compute Q targets for current states
    q_targets = rewards_batch + discount_factor * next_q_targets * (1 - dones_batch)

    # Get expected Q values from local model
    q_expected = self.local_qnetwork(states_batch).gather(1, actions_batch)

    # Compute loss
    loss = F.mse_loss(q_expected, q_targets)
    # Minimize the loss
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()

### Initializing the DCQN agent

In [ ]:
agent = Agent(number_actions)

### Training the DCQN agent

In [ ]:
number_episodes = 2000 # no. of times the agent does the test
maximum_number_timesteps_per_episode = 10000 # max no. of timesteps per episode
epsilon_starting_value  = 1.0
epsilon_ending_value  = 0.01
epsilon_decay_value  = 0.995
epsilon = epsilon_starting_value
scores_on_100_episodes = deque(maxlen = 100)

for episode in range(1, number_episodes + 1):
  state, _ = env.reset()
  score = 0
  for t in range(maximum_number_timesteps_per_episode):
    action = agent.act(state, epsilon)
    next_state, reward, done, _, _ = env.step(action)
    agent.step(state, action, reward, next_state, done)
    state = next_state
    score += reward
    if done:
      break
  scores_on_100_episodes.append(score)
  epsilon = max(epsilon_ending_value, epsilon_decay_value * epsilon)
  print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)), end = "")
  if episode % 100 == 0:
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)))
  if np.mean(scores_on_100_episodes) >= 500.0:
    print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(episode - 100, np.mean(scores_on_100_episodes)))
    torch.save(agent.local_qnetwork.state_dict(), 'checkpoint.pth')
    break

### Visualizing the results

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)
        state, reward, done, _, _ = env.step(action)
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'MsPacmanNoFrameskip-v0') # The MsPacman environment was renamed 'MsPacmanNoFrameskip-v0'

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()